In [1]:
import open_clip

In [2]:
import numpy as np
import pandas as pd

In [3]:
# pretrained also accepts local paths
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k') 

open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

C:\Users\dc22948\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dc22948\.cache\huggingface\hub\models--laion--CLIP-ViT-B-32-laion2B-s34B-b79K. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [4]:
model.eval()
#token 数
context_length = model.context_length
#clip可以识别的唯一单词或符号的数量。
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Context length:", context_length)
print("Vocab size:", vocab_size)

Model parameters: 151,277,313
Context length: 77
Vocab size: 49408


In [5]:
preprocess

Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    <function _convert_to_rgb at 0x0000018260AEF6A0>
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)

In [6]:
from open_clip import tokenizer

In [7]:
import os
import skimage
import IPython.display
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

from collections import OrderedDict
import torch

%matplotlib inline
%config InlineBackend.figure_format = 'retina'


# dealing with the sketch dataset

In [13]:
import zipfile
import os

# 设置 zip 文件路径
zip_path = 'ImageNet/ImageNet-Sketch/ImageNet-Sketch.zip'  # 注意：确保扩展名是 .zip

# 设置解压目标路径
extract_path = 'ImageNet/ImageNet-Sketch'

# 解压文件
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("解压完成！")


解压完成！


In [14]:
from torch.utils.data import Dataset
from PIL import Image
import os

class ImageNetSketchDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        for root, _, files in os.walk(root_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(root, file))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image


In [15]:
dataset = ImageNetSketchDataset(
    root_dir='ImageNet/ImageNet-Sketch/sketch',
    transform=preprocess
)

In [16]:
import os

# 设置解压后的图像根目录
image_root = 'ImageNet/ImageNet-Sketch/sketch'  # 根据你的实际解压路径调整

# 支持的图像扩展名
image_extensions = ('.png', '.jpg', '.jpeg', '.bmp')

# 统计图像数量
image_count = 0
for root, _, files in os.walk(image_root):
    for file in files:
        if file.lower().endswith(image_extensions):
            image_count += 1

print(f"解压后的图像总数：{image_count} 张")


解压后的图像总数：50889 张


In [13]:
# 文件名: extract_logits_from_sketch.py

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()


# extracting the logits

In [18]:

# ============ 包装 Dataset，确保图像转为 Tensor ============
class ImageNetSketchDataset(Dataset):
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset
        self.to_tensor = transforms.ToTensor()

In [19]:



    def __getitem__(self, index):
        img, label = self.base_dataset[index]
        if isinstance(img, np.ndarray):
            img = Image.fromarray(img)
        if isinstance(img, Image.Image):
            img = self.to_tensor(img)
        return img, label


    def __len__(self):
        return len(self.base_dataset)


    @property
    def targets(self):
        return self.base_dataset.targets

##  using text feature 

In [22]:
import torch

# 加载类别映射 pt 文件（类别ID -> 名称）
class_dict = torch.load("ImageNet/classnames.pt")
class_names = list(class_dict.values())

# 构造文本描述
text_descriptions = [f"A sketch of a {name}" for name in class_names]

# Tokenize 并送入模型
text_tokens = tokenizer.tokenize(text_descriptions).to(device)

with torch.no_grad():
    text_features = model.encode_text(text_tokens).float()
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

# 可选：保存特征
text_features_np = text_features.cpu().numpy()


## using image feature

In [23]:
from tqdm import tqdm
import torch.nn.functional as F

def extract_image_features(dataset, name="Sketch"):
    features = []
    for img_tensor in tqdm(dataset, desc=f"[{name}] 提取图像特征"):
        resized = F.interpolate(img_tensor.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False)
        img_input = resized.to(device)
        with torch.no_grad():
            image_feature = model.encode_image(img_input).float()
            image_feature = image_feature / image_feature.norm(dim=-1, keepdim=True)
        features.append(image_feature.squeeze(0).cpu())
    return torch.stack(features).numpy()

# 提取图像特征
sketch_features = extract_image_features(dataset)


[Sketch] 提取图像特征: 100%|██████████| 50889/50889 [37:25<00:00, 22.66it/s]  


## calculating the logits and storaging

In [24]:
import numpy as np
import pickle
import os

temperature = 100.0
logits = temperature * sketch_features @ text_features_np.T  # [N x C]
labels = np.array([i for i in range(len(sketch_features))]).reshape(-1, 1)  # 或你自己的标签格式

# 保存为 .p 文件
output_data = (logits, labels)
output_dir = "logits_sketch"
os.makedirs(output_dir, exist_ok=True)
save_path = os.path.join(output_dir, "sketch_logits.p")

with open(save_path, "wb") as f:
    pickle.dump(output_data, f)

print(f"Logits 计算完成，保存路径：{save_path}")

Logits 计算完成，保存路径：logits_sketch\sketch_logits.p


## injunktion ontwo .p documents

In [ ]:
# import pickle

# # 加载 train 数据
# with open("logits_sketch/train_logits.p", "rb") as f:
#     train_logits, train_labels = pickle.load(f)

# # 加载 test 数据
# with open("logits_sketch/test_logits.p", "rb") as f:
#     test_logits, test_labels = pickle.load(f)

# # 合并为嵌套结构
# merged_data = ((train_logits, train_labels), (test_logits, test_labels))

# # 保存合并结果
# save_path = "logits_sketch/sketch_logits_combined.p"
# with open(save_path, "wb") as f:
#     pickle.dump(merged_data, f)

# print(f"已合并并保存为：{save_path}")

# dealing with A dataset

In [26]:
import tarfile
import os

tar_path = 'ImageNet/ImageNet-A/ImageNet-a.tar'
extract_path = 'ImageNet/ImageNet-A/images'

# 解压 tar 文件
with tarfile.open(tar_path, 'r') as tar:
    tar.extractall(path=extract_path)

print("解压完成！图像已提取到：", extract_path)


C:\Users\dc22948\AppData\Local\Temp\ipykernel_10580\2109190124.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


解压完成！图像已提取到： ImageNet/ImageNet-A/images


In [28]:
import torch

class_dict = torch.load("ImageNet/ImageNet-A/classnames-a.pt")
class_to_idx = {cls_id: idx for idx, cls_id in enumerate(class_dict.keys())}


## extracting the logits

### using text feature

In [29]:

text_descriptions = [f"A sketch of a {name}" for name in class_dict.values()]

# Tokenize 并编码为文本特征
text_tokens = tokenizer.tokenize(text_descriptions).to(device)
with torch.no_grad():
    text_features = model.encode_text(text_tokens).float()
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

text_features_np = text_features.cpu().numpy()


### using image feature

In [33]:
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms
import os

class ImageNetALabeledDataset(Dataset):
    def __init__(self, root_dir, transform=None, class_to_idx=None):
        self.root_dir = root_dir
        self.transform = transform
        self.class_to_idx = class_to_idx
        self.image_labels = []

        for root, _, files in os.walk(root_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    cls_id = os.path.basename(root)
                    label_index = self.class_to_idx.get(cls_id, -1)
                    if label_index >= 0:
                        self.image_labels.append((os.path.join(root, file), label_index))

    def __getitem__(self, idx):
        path, label = self.image_labels[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

    def __len__(self):
        return len(self.image_labels)

In [34]:
preprocess

Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    <function _convert_to_rgb at 0x000002559868DEE0>
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)

In [35]:
dataset = ImageNetALabeledDataset(
    root_dir='ImageNet/ImageNet-A/images',
    transform=preprocess,
    class_to_idx=class_to_idx
)

labels = np.array([dataset[i][1] for i in range(len(dataset))]).reshape(-1, 1)


In [37]:
import torch.nn.functional as F
from tqdm import tqdm

def extract_image_features(dataset, name="ImageNet-A"):
    features = []
    for img_tensor, _ in tqdm(dataset, desc=f"[{name}] 提取图像特征"):
        resized = F.interpolate(img_tensor.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False)
        img_input = resized.to(device)
        with torch.no_grad():
            image_feature = model.encode_image(img_input).float()
            image_feature = image_feature / image_feature.norm(dim=-1, keepdim=True)
        features.append(image_feature.squeeze(0).cpu())
    return torch.stack(features).numpy()

image_features = extract_image_features(dataset)

[ImageNet-A] 提取图像特征: 100%|██████████| 7500/7500 [04:45<00:00, 26.28it/s]


### calcuclating the logits and storanging

In [40]:
# # 提取图像特征（如上一轮的 extract_image_features）
# # image_features = extract_image_features(dataset)

# # 提取标签
# labels = np.array([dataset[i][1] for i in range(len(dataset))]).reshape(-1, 1)

# 计算 logits
temperature = 100.0
image_logits = temperature * sketch_features @ text_features_np.T

# 保存为测试集格式 (test_logits, test_labels)
test_data = (image_logits, labels)
with open("imagea.p", "wb") as f:
    pickle.dump(test_data, f)

print("测试集 logits 已保存：imagea.p")


测试集 logits 已保存：imagea.p


In [45]:
print("顶层类型：", type(data))
print("元素数量：", len(data))
for i, item in enumerate(data):
    print(f"第 {i} 项类型: {type(item)}")


顶层类型： <class 'tuple'>
元素数量： 2
第 0 项类型: <class 'numpy.ndarray'>
第 1 项类型: <class 'numpy.ndarray'>


# dealing with v2 dataset

In [47]:
tar_path = 'imageNet/ImageNet-V2/imagenetv2.tar.gz'
extract_path = 'ImageNet/ImageNet-V2/images'

# 解压 tar 文件
with tarfile.open(tar_path, 'r') as tar:
    tar.extractall(path=extract_path)

print("解压完成！图像已提取到：", extract_path)


C:\Users\dc22948\AppData\Local\Temp\ipykernel_10580\2692615686.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


解压完成！图像已提取到： ImageNet/ImageNet-V2/images


In [57]:
class_dict_V2 = torch.load("ImageNet/ImageNet-V2/classnames-v2.pt")
class_to_idx_V2 = {cls_id: idx for idx, cls_id in enumerate(class_dict_V2.keys())}


## extracting the logits

### using text feature

In [61]:
text_descriptions_V2 = [f"A photo of a {name}" for name in class_dict_V2.values()]

# Tokenize 并编码为文本特征
text_tokens_V2 = tokenizer.tokenize(text_descriptions_V2).to(device)
with torch.no_grad():
    text_features_V2 = model.encode_text(text_tokens_V2).float()
    text_features_V2 = text_features_V2 / text_features_V2.norm(dim=-1, keepdim=True)

text_features_np_V2 = text_features_V2.cpu().numpy()


### using image feature

In [62]:
class ImageNetv2LabeledDataset(Dataset):
    def __init__(self, root_dir_V2, transform=None, class_to_idx_V2=None):
        self.root_dir = root_dir_V2
        self.transform = transform
        self.class_to_idx = class_to_idx_V2
        self.image_labels_V2 = []

        for root_V2, _, files in os.walk(root_dir_V2):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    cls_id_V2 = os.path.basename(root_V2)
                    label_index_V2 = self.class_to_idx.get(cls_id_V2, -1)
                    if label_index_V2 >= 0:
                        self.image_labels_V2.append((os.path.join(root_V2, file), label_index_V2))

    def __getitem__(self, idx):
        path, label_V2 = self.image_labels_V2[idx]
        image_V2 = Image_V2.open(path).convert("RGB")
        if self.transform:
            image_V2 = self.transform(image_V2)
        return image_V2, label_V2

    def __len__(self):
        return len(self.image_labels_V2)

In [63]:
dataset_V2 = ImageNetv2LabeledDataset(
    root_dir_V2='ImageNet/ImageNet-V2/images',
    transform=preprocess,
    class_to_idx_V2=class_to_idx_V2
)

labels_V2 = np.array([dataset_V2[i][1] for i in range(len(dataset_V2))]).reshape(-1, 1)


In [66]:
def extract_image_features(dataset_V2, name="ImageNet-A"):
    features = []
    for img_tensor, _ in tqdm(dataset_V2, desc=f"[{name}] 提取图像特征"):
        resized = F.interpolate(img_tensor.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False)
        img_input = resized.to(device)
        with torch.no_grad():
            image_feature_V2 = model.encode_image(img_input).float()
            image_feature_V2 = image_feature_V2 / image_feature_V2.norm(dim=-1, keepdim=True)
        features.append(image_feature_V2.squeeze(0).cpu())
    return torch.stack(features).numpy()

image_features_V2 = extract_image_features(dataset_V2)

[ImageNet-A] 提取图像特征: 0it [00:00, ?it/s]


RuntimeError: stack expects a non-empty TensorList

In [8]:
import tarfile
import os

tar_path = 'ImageNet/Test/test_images.tar.gz'
extract_path = 'ImageNet/Test/images'
os.makedirs(extract_path, exist_ok=True)

with tarfile.open(tar_path, 'r') as tar:
    tar.extractall(path=extract_path)

print("解压完成：", extract_path)


C:\Users\dc22948\AppData\Local\Temp\ipykernel_4360\1908485041.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


解压完成： ImageNet/Test/images


In [9]:
import torch

class_dict_test = torch.load("ImageNet/Test/classnames.pt")
class_to_idx_test = {cls_id: idx for idx, cls_id in enumerate(class_dict_test.keys())}


In [22]:
from torch.utils.data import Dataset
from PIL import Image
import os

class ImageNetTestFilenameLabeledDataset(Dataset):
    def __init__(self, image_dir, transform=None, class_to_idx=None):
        self.image_dir = image_dir
        self.transform = transform
        self.class_to_idx = class_to_idx
        self.image_labels = []

        for fname in os.listdir(image_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                parts = fname.split('_')
                # 提取 nXXXXXX 类别 ID
                class_id = parts[-1].split('.')[0] if len(parts) > 1 else None
                label = self.class_to_idx.get(class_id, -1)
                if label >= 0:
                    img_path = os.path.join(image_dir, fname)
                    self.image_labels.append((img_path, label))

    def __getitem__(self, idx):
        path, label = self.image_labels[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

    def __len__(self):
        return len(self.image_labels)


In [24]:
dataset_test = ImageNetTestFilenameLabeledDataset(
    image_dir='ImageNet/Test/images',
    transform=preprocess,
    class_to_idx=class_to_idx_test  # 从 classnames-test.pt 构建的字典
)

labels_test = np.array([dataset_test[i][1] for i in range(len(dataset_test))]).reshape(-1, 1)


In [28]:
text_descriptions_test = [f"A photo of a {name}" for name in class_dict_test.values()]
text_tokens_test = tokenizer.tokenize(text_descriptions_test).to(device)

with torch.no_grad():
    text_features_test = model.encode_text(text_tokens_test).float()
    text_features_test = text_features_test / text_features_test.norm(dim=-1, keepdim=True)

text_features_np_test = text_features_test.cpu().numpy()


In [ ]:
import torch.nn.functional as F
from tqdm import tqdm

def extract_image_features_test(dataset, model, device, name="TestSet"):
    features = []
    failed_count = 0

    print(f"开始提取 {len(dataset)} 张图像特征")

    for i, (img_tensor, _) in enumerate(tqdm(dataset, desc=f"[{name}] 提取图像特征")):
        try:
            if img_tensor.ndim == 3:
                img_tensor = img_tensor.unsqueeze(0)

            resized = F.interpolate(img_tensor, size=(224, 224), mode='bilinear', align_corners=False)
            img_input = resized.to(device)

            with torch.no_grad():
                image_feature = model.encode_image(img_input).float()
                image_feature = image_feature / image_feature.norm(dim=-1, keepdim=True)

            features.append(image_feature.squeeze(0).cpu())
        except Exception as e:
            failed_count += 1
            print(f"第 {i} 张图像处理失败：{e}")

    if not features:
        raise ValueError("没有成功提取任何图像特征")

    print(f"成功提取 {len(features)} 张图像特征，失败数：{failed_count}")
    return torch.stack(features).numpy()


In [ ]:
image_features_test = extract_image_features_test(dataset_test, model, device)
logits_test = 100.0 * image_features_test @ text_features_np_test.T

import pickle
os.makedirs("logits_test", exist_ok=True)

with open("logits_test/test_logits.p", "wb") as f:
    pickle.dump((logits_test, labels_test), f)

print("Logits 和标签已保存到：logits_test/test_logits.p")


In [ ]:
import numpy as np
import pickle
import os

# 提取 logits：CLIP 风格计算
logits_test = 100.0 * image_features_test @ text_features_np_test.T

# 提取 labels：从文件名中解析出 nXXXXXX 类别 ID
def extract_class_id(fname):
    parts = fname.split('_')
    return parts[-1].split('.')[0] if len(parts) > 1 else "unknown"

# 获取 filenames（作为标签来源）
filenames = [dataset_test[i][1] for i in range(len(dataset_test))]
labels_test = np.array([extract_class_id(fname) for fname in filenames]).reshape(-1, 1)

# 创建保存目录
os.makedirs("logits_test", exist_ok=True)
save_path = "logits_test/test_logits.p"

# 保存为标准 .p 文件结构：（logits, labels）
with open(save_path, "wb") as f:
    pickle.dump((logits_test, labels_test), f)

print(f"已保存 logits 和 labels 到：{save_path}")
print("logits shape:", logits_test.shape)
print("labels shape:", labels_test.shape)


In [27]:
print("数据集大小：", len(dataset_test))


数据集大小： 0


In [18]:
import os

root_dir = 'ImageNet/Test/images'
print("子文件夹示例：", os.listdir(root_dir)[:5])


子文件夹示例： ['ILSVRC2012_test_00000001.JPEG', 'ILSVRC2012_test_00000002.JPEG', 'ILSVRC2012_test_00000003.JPEG', 'ILSVRC2012_test_00000004.JPEG', 'ILSVRC2012_test_00000005.JPEG']


In [21]:
import torch

# 加载类名映射文件
class_dict = torch.load("ImageNet/Test/classnames.pt")  # 根据你的命名改路径

# 打印前 10 个 key-value 对
print("前十个类别映射：")
for i, (cls_id, cls_name) in enumerate(class_dict.items()):
    print(f"{i+1:2d}. 类别ID: {cls_id} → 类名: {cls_name}")
    if i >= 9:
        break


前十个类别映射：
 1. 类别ID: n01440764 → 类名: tench
 2. 类别ID: n01443537 → 类名: goldfish
 3. 类别ID: n01484850 → 类名: great white shark
 4. 类别ID: n01491361 → 类名: tiger shark
 5. 类别ID: n01494475 → 类名: hammerhead
 6. 类别ID: n01496331 → 类名: electric ray
 7. 类别ID: n01498041 → 类名: stingray
 8. 类别ID: n01514668 → 类名: cock
 9. 类别ID: n01514859 → 类名: hen
10. 类别ID: n01518878 → 类名: ostrich
